# 02 — LexRank (estrattivo)

**LexRank** (Erkan & Radev, 2004) è, come TextRank, un metodo estrattivo basato su grafi e
PageRank; la differenza sta nella rappresentazione delle frasi: qui la similarità è la cosine
similarity tra vettori **TF-IDF** (niente reti neurali, niente download di modelli), il che lo
rende molto più veloce di TextRank su CPU. Il riassunto è composto dalle `N_SENTENCES` frasi con
punteggio più alto, nella loro forma originale.

Come il notebook 01, opera su due ambiti (`SCOPE`):
- **`sample`** — il campione condiviso di [00_prepara_campione.ipynb](00_prepara_campione.ipynb),
  per il confronto con i metodi abstractive;
- **`full`** — l'intero dataset pulito (`data/tab/complete.tab`, 56.101 esempi, in streaming),
  per il confronto tra estrattivi. Essendo TF-IDF puro, la corsa completa è fattibile anche su
  CPU (ore, non giorni).

Riassunti salvati incrementalmente in `results/summaries/lexrank_{scope}.tsv` con **ripresa**
automatica dopo un'interruzione; metriche ricalcolabili dai soli file salvati.

In [1]:
# Installa le dipendenze se mancanti (per esempio su Google Colab)
try:
    import pyAutoSummarizer  # noqa: F401
except ImportError:
    %pip install pyAutoSummarizer sentencepiece

In [1]:
# --- Configurazione ---------------------------------------------------------
import summ_utils as su

METODO      = 'lexrank'
SCOPE       = 'full'   # 'sample' = campione condiviso | 'full' = intero complete.tab
N_SAMPLES   = 100        # deve combaciare con il campione creato dal notebook 00
SEED        = 42
LIMIT       = None       # es. 3 per uno smoke test rapido; None = tutti
N_SENTENCES = 11         # frasi estratte per riassunto (mediana del corpus: 11 frasi/riassunto)
STOP_WORDS  = ['en']     # pre-processing per il ranking (l'output resta il testo originale)

BASE   = su.trova_base_dir()
P      = su.percorsi_standard(BASE)
SAMPLE_PATH = P['sample_dir'] / f'sample_{N_SAMPLES}_seed{SEED}.tsv'
OUT_PATH    = P['summaries_dir'] / f'{METODO}_{SCOPE}.tsv'

print(f'Ambito (SCOPE) : {SCOPE}')
print(f'Output         : {OUT_PATH}')

Ambito (SCOPE) : full
Output         : c:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\summaries\lexrank_full.tsv


## Generazione dei riassunti

Per ogni documento: separatore `` ||||| `` sostituito con un newline, istanza `psr.summarization`
(il pre-processing serve solo al ranking), rank con `summ_lex_rank` (TF-IDF + PageRank) ed
estrazione delle prime `N_SENTENCES` frasi con `show_summary`. Nessun modello da caricare.

In [2]:
from pyAutoSummarizer.base import psr

def genera(documento):
    s = psr.summarization(documento, stop_words=STOP_WORDS)
    rank = s.summ_lex_rank()
    return s.show_summary(rank, n=N_SENTENCES)

esempi = (su.carica_campione(SAMPLE_PATH) if SCOPE == 'sample'
          else su.itera_complete_tab(P['complete_tab']))
scrittore = su.ScrittoreRiassunti(OUT_PATH)
errori = su.ciclo_summarization(esempi, scrittore, genera, limit=LIMIT, etichetta='LexRank ')
scrittore.chiudi()

c:\Users\antonio.girasella\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LexRank [1] media 0.0 s/esempio (saltati 0 gia' fatti)
LexRank [2] media 0.0 s/esempio (saltati 0 gia' fatti)
LexRank [3] media 0.0 s/esempio (saltati 0 gia' fatti)
LexRank [10] media 0.0 s/esempio (saltati 0 gia' fatti)
LexRank [20] media 0.0 s/esempio (saltati 0 gia' fatti)
LexRank [30] media 0.0 s/esempio (saltati 0 gia' fatti)
LexRank [40] media 0.0 s/esempio (saltati 0 gia' fatti)
LexRank [50] media 0.0 s/esempio (saltati 0 gia' fatti)
LexRank [60] media 0.0 s/esempio (saltati 0 gia' fatti)
LexRank [70] media 0.0 s/esempio (saltati 0 gia' fatti)
LexRank [80] media 0.0 s/esempio (saltati 0 gia' fatti)
LexRank [90] media 0.0 s/esempio (saltati 0 gia' fatti)
LexRank [100] media 0.0 s/esempio (saltati 0 gia' fatti)
LexRank [110] media 0.0 s/esempio (saltati 0 gia' fatti)
LexRank [120] media 0.0 s/esempio (saltati 0 gia' fatti)
LexRank [130] media 0.0 s/esempio (saltati 0 gia' fatti)
LexRank [140] media 0.0 s/esempio (saltati 0 gia' fatti)
  ERRORE su row_id=143: IndexError('list assig

## Valutazione (indipendente dalla generazione)

Legge **solo** i file salvati; rieseguibile senza rigenerare i riassunti. Metriche ROUGE-1/2/L
(F1, precisione, recall), BLEU e METEOR con normalizzazione identica per tutti i metodi del
benchmark. Output: `results/metrics/lexrank_{scope}_per_example.csv` e `…_aggregate.json`
(medie complessive e per split).

In [3]:
import json

riassunti   = su.carica_riassunti(OUT_PATH)
riferimenti = (su.carica_campione(SAMPLE_PATH) if SCOPE == 'sample'
               else su.itera_complete_tab(P['complete_tab']))
config = {'n_sentences': N_SENTENCES, 'stop_words': STOP_WORDS,
          'libreria': 'pyAutoSummarizer 1.2.0'}

righe, aggregato = su.valuta_e_salva(riferimenti, riassunti, METODO, SCOPE,
                                     P['metrics_dir'], config)
print(json.dumps(aggregato['overall'], indent=2))

Metriche per-esempio : c:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\metrics\lexrank_full_per_example.csv (55894 righe)
Metriche aggregate   : c:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\metrics\lexrank_full_aggregate.json
{
  "rouge1_f1": 0.3641898600796998,
  "rouge1_p": 0.30772693360527015,
  "rouge1_r": 0.4794515117262006,
  "rouge2_f1": 0.13693961996333814,
  "rouge2_p": 0.11060675231629169,
  "rouge2_r": 0.19685835489753678,
  "rougeL_f1": 0.16951487355403122,
  "rougeL_p": 0.13039193988622505,
  "rougeL_r": 0.26585075464664293,
  "bleu": 0.0835910617610864,
  "meteor": 0.5022497381108285,
  "parole_generate": 449.5351558306795
}


## Ispezione qualitativa

In [4]:
riferimenti = (su.carica_campione(SAMPLE_PATH) if SCOPE == 'sample'
               else su.itera_complete_tab(P['complete_tab']))
su.mostra_esempi(riferimenti, riassunti, quanti=2)

--- row_id=0 (split=train) ---
GENERATO   : A fresh update on the U.S. employment situation for January hits the wires at 8:30 a.m. New York time offering one of the most important snapshots on how the economy fared during the previous month The economy has added 858,000 jobs since December _ the best four months of hiring in two years The Labor Department says the economy added 120,000 jobs in March, down from more than 200,000 in each of the previous three months Employers pulled back sharply on hiring last month, a reminder that the U.
RIFERIMENTO: The unemployment rate dropped to 8.2% last month, but the economy only added 120,000 jobs, when 203,000 new jobs had been predicted, according to today's jobs report. Reaction on the Wall Street Journal's MarketBeat Blog was swift: "Woah!!! Bad number." The unemployment rate, however, is better news; it had been expected to hold steady at 8.3%. But the AP notes that the dip is mostly due to more Americans giving up on seeking employment.
